# 04 — Central Catalogue

Demonstrates the server-side workflow:

1. **Ingest** — push a local catalogue into the pgSTAC database.
2. **Browse** — open STAC Browser at `http://localhost:8000` to navigate the catalogue visually.
3. **Query** — pull a subset back down to a local catalogue with `eomatch-query`.
4. **Analyse** — open the queried catalogue with `MatchupCatalogue` and inspect the result.

**Prerequisites:**
- Run `01_find_and_catalogue.ipynb` so that `example/catalogue/` exists.
- The Docker Compose stack running: `cd deploy && docker compose up -d`
- `eomatch[ingest]` and `eomatch[query]` extras installed:
  ```
  pip install -e '.[ingest,query]'
  ```

Connection parameters below assume the default `docker compose` setup.
Adjust `DB_*` variables for a remote server.

In [ ]:
import datetime as dt
import logging
import os

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-8s %(name)s: %(message)s",
    datefmt="%H:%M:%S",
)

NOTEBOOK_DIR = os.path.dirname(os.path.abspath("__file__"))
CATALOGUE_DIR   = os.path.join(NOTEBOOK_DIR, "catalogue")   # produced by notebook 01
QUERIED_DIR     = os.path.join(NOTEBOOK_DIR, "queried")      # written by eomatch-query

# ---- Server / API config -------------------------------------------------------
API_URL     = "http://localhost:8000/external/"
DB_HOST     = "localhost"
DB_PORT     = 5432
DB_NAME     = "eomatch"
DB_USER     = "postgres"
DB_PASSWORD = os.environ.get("PGPASSWORD", "eomatch")

print("Local catalogue:", CATALOGUE_DIR)
print("API URL:        ", API_URL)

## Step 1 — Ingest the local catalogue

`eomatch.ingest.ingest` reads every collection and item from the local catalogue and
upserts them into the pgSTAC database.  Running it more than once is safe — items are
replaced if they already exist.

`assets_base_url` rewrites thumbnail hrefs from `file://` paths to HTTP URLs so STAC
Browser can load them.  Leave it as `None` if your items have no local-file assets.

In [ ]:
from eomatch.ingest import ingest

ingest(
    catalogue_path=CATALOGUE_DIR,
    db_host=DB_HOST,
    db_port=DB_PORT,
    db_name=DB_NAME,
    db_user=DB_USER,
    db_password=DB_PASSWORD,
    assets_base_url="http://localhost:8000/catalogue",
)

print("Ingest complete.")

## Step 2 — Browse the catalogue

Open [http://localhost:8000](http://localhost:8000) in your browser.  STAC Browser connects
to the `/external/` API endpoint (configured in `deploy/browser/config.js`) and lets you
navigate collections, inspect item metadata, and follow `derived_from` links between
matchup items and their source products.

> The browser URL opens automatically in some Jupyter environments; otherwise click the
> link that is printed below.

In [ ]:
from IPython.display import display, Markdown
display(Markdown(f"**STAC Browser:** [{API_URL.replace('/external/', '')}]({API_URL.replace('/external/', '')})"))

## Step 3 — Query the central catalogue

`eomatch.query.query` searches the STAC API and saves the results as a local catalogue
identical in structure to one produced by `find_and_catalogue`.  Items referenced via
`derived_from` / `related` links are fetched automatically so that `get_events()` works
without network access.

Adjust `collections`, `start_time`, `end_time`, and `bbox` to pull a subset.

In [ ]:
from eomatch.query import query

query(
    api_url=API_URL,
    output_path=QUERIED_DIR,
    # collections=["LANDSAT_C2L1-Landsat-9-vs-S2_MSI_L1C-S2A"],  # optional filter
    start_time=dt.datetime(2022, 1, 1),
    end_time=dt.datetime(2022, 12, 31),
)

print("Query complete. Local copy written to:", QUERIED_DIR)

## Step 4 — Open and inspect the queried catalogue

The queried catalogue is a normal local pystac catalogue — open it with `MatchupCatalogue`
exactly as in `02_explore_catalogue.ipynb`.  Item links have been converted back from
absolute HTTP URLs to relative filesystem paths, so `get_events()` works offline.

In [ ]:
from eomatch import MatchupCatalogue

queried_cat = MatchupCatalogue.open(QUERIED_DIR)

print("Collections in queried catalogue:")
for col in queried_cat.catalog.get_children():
    items = list(col.get_items())
    print(f"  [{col.id}]  ({len(items)} item(s))")

In [ ]:
events = queried_cat.get_events()
print(f"Events recovered: {len(events)}")

for event in events:
    n = len(event.matchup_set) if event.matchup_set is not None else 0
    print(f"\n  {event.stac_id}")
    print(f"  Platforms:   {event.platforms}")
    print(f"  Matchups:    {n}")
    if event.matchup_set:
        for mu in event.matchup_set:
            print(f"    time diff (s): {mu.time_diff_abs:.1f}")
            for p in mu.products:
                print(f"    [{p.collection}]  {p.id}")

## How links work across the stack

| Layer | Link form | Set by |
|---|---|---|
| Local catalogue | Relative filesystem paths (`../../../../LANDSAT_C2L1/…`) | pystac `save()` |
| pgSTAC | Root-relative API paths (`/api/collections/{id}/items/{id}`) | `ingest._rewrite_item_links` (in memory only) |
| stac-fastapi (`/api/`) | Absolute HTTP URLs (`http://host/api/collections/…`) | stac-fastapi at serve time |
| Filter-proxy (`/external/`) | `/api/` → `/external/` rewrite | `deploy/filter-proxy/main.py` |
| After `eomatch-query` | Relative filesystem paths again | `query._rewrite_cross_item_links` |

The local catalogue JSON files are **never** modified by ingest or query — the rewrites
are always in-memory or applied to a fresh copy.